# 01 — Vicon CSV → Quantity of Motion (QdM)

This notebook:
- reads Vicon "Trajectories" CSV exports (3D markers)
- computes Movement Quantity Index (MQI) as cumulative frame-to-frame 3D displacement (mm)
- extracts wrists and head (temples) for P1 and P2
- exports an Excel summary table

**Author:** Matys Précloux  
**Project:** SYNCOGEST

In [5]:
import sys
from pathlib import Path
import pandas as pd

from src_vicon_utils import read_vicon_csv, find_xyz_cols, motion_quantity_point

## 1) Project paths (MODE-aware)

We support two modes:
- `test`: small dataset committed to GitHub
- `raw`: full dataset (typically ignored)

Input:
- `data/<MODE>/vicon_csv/*.csv`

Output:
- `results/<MODE>/vicon_MQI_wrists_head_mm.xlsx`


In [6]:

# Project root + import path
# Detect project root (works when executed inside / notebooks)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Ensure project root is importable (for local modules)
sys.path.insert(0, str(PROJECT_ROOT))

# Mode (test vs raw) + paths
import os
MODE = os.environ.get("MODE", "test")

# Input and output folders
ROOT = PROJECT_ROOT / "data" / MODE / "vicon_csv"
RESULTS_DIR = PROJECT_ROOT / "results" / MODE
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Mode:", MODE)
print("Project root:", PROJECT_ROOT)
print("Vicon CSV root:", ROOT)
print("ROOT exists:", ROOT.exists())
print("Results dir:", RESULTS_DIR.resolve())

Mode: test
Project root: /Users/matysprecloux/Desktop/Master IEAP/Code MOTTET/Defense /SYNCOGESTM2
Vicon CSV root: /Users/matysprecloux/Desktop/Master IEAP/Code MOTTET/Defense /SYNCOGESTM2/data/test/vicon_csv
ROOT exists: True
Results dir: /Users/matysprecloux/Desktop/Master IEAP/Code MOTTET/Defense /SYNCOGESTM2/results/test


## 2) Core processing function

`process_one_csv()` reads one Vicon CSV and returns:
- per-marker QdM for wrists and temples (mm)
- aggregated indices:
  - `P1_MQI_WRISTS_mm`, `P2_MQI_WRISTS_mm`
  - `P1_MQI_HEAD_mm`, `P2_MQI_HEAD_mm`

In [7]:

# 4) Process one CSV file
# For each file, we want to compute MQI for wrists and head, and store results in a dict.
def process_one_csv(csv_path: Path):
    """
    Process a single Vicon CSV file and compute MQI (mm) for:
    - wrists (left/right)
    - head (left/right temples)

    Returns
    -------
    dict
        One-row summary for this file (ready to become a DataFrame row).
    """
    df = read_vicon_csv(csv_path)
    cols = list(df.columns)

    out = {"csv": str(csv_path), "file": csv_path.name}
# Marker tokens for each participant
    subjects = {
        "P1": {
            "WR_D": "poignet_D",
            "WR_G": "poignet_G",
            "TP_D": "Tempe_D",
            "TP_G": "Tempe_G",
        },
        "P2": {
            "WR_D": "2poignet_D",
            "WR_G": "2poignet_G",
            "TP_D": "2Tempe_D",
            "TP_G": "2Temps_G",
        }
    }
    # For each participant, find the relevant columns and compute MQI for wrists and head
    for pid, tok in subjects.items():

        # Wrists 
        # For each wrist (right/left), find the relevant columns and compute MQI
        for side_key in ["WR_D", "WR_G"]:
            token = tok[side_key]
            X, Y, Z = find_xyz_cols(cols, token)
            # Check if all required columns were found
            if None in (X, Y, Z):
                out[f"{pid}_{token}_error"] = "missing X/Y/Z"
            else:
                q, nsteps = motion_quantity_point(df, X, Y, Z)
                out[f"{pid}_{token}_MQI_mm"] = q
                out[f"{pid}_{token}_nsteps"] = nsteps
        # Aggregate wrists (if both sides are available)
        wd = out.get(f"{pid}_{tok['WR_D']}_MQI_mm")
        wg = out.get(f"{pid}_{tok['WR_G']}_MQI_mm")

        if wd is not None and wg is not None:
            out[f"{pid}_MQI_WRISTS_mm"] = wd + wg

        # Head 
        for side_key in ["TP_D", "TP_G"]:
            token = tok[side_key]
            X, Y, Z = find_xyz_cols(cols, token)

            if None in (X, Y, Z):
                out[f"{pid}_{token}_error"] = "missing X/Y/Z"
            else:
                q, nsteps = motion_quantity_point(df, X, Y, Z)
                out[f"{pid}_{token}_MQI_mm"] = q
                out[f"{pid}_{token}_nsteps"] = nsteps

        td = out.get(f"{pid}_{tok['TP_D']}_MQI_mm")
        tg = out.get(f"{pid}_{tok['TP_G']}_MQI_mm")

        if td is not None and tg is not None:
            out[f"{pid}_MQI_HEAD_mm"] = td + tg

    return out

## 3) Batch processing and export

We iterate over all CSV files in `data/<MODE>/vicon_csv/`,
compute MQI summaries, and export an Excel table.

In [8]:

# 5) Batch processing

csv_files = sorted(ROOT.rglob("*.csv"))
print("CSV found:", len(csv_files))

rows = []

for f in csv_files:
    try:
        rows.append(process_one_csv(f))
    except Exception as e:
        rows.append({"csv": str(f), "file": f.name, "error": repr(e)})

df_out = pd.DataFrame(rows)

out_path = RESULTS_DIR / "vicon_MQI_wrists_head_mm.xlsx"
df_out.to_excel(out_path, index=False)

print("Saved:", out_path.resolve())
df_out.head()

CSV found: 4
Saved: /Users/matysprecloux/Desktop/Master IEAP/Code MOTTET/Defense /SYNCOGESTM2/results/test/vicon_MQI_wrists_head_mm.xlsx


,csv,file,P1_poignet_D_MQI_mm,P1_poignet_D_nsteps,P1_poignet_G_MQI_mm,P1_poignet_G_nsteps,P1_MQI_WRISTS_mm,P1_Tempe_D_MQI_mm,P1_Tempe_D_nsteps,P1_Tempe_G_MQI_mm,...,P2_2poignet_D_MQI_mm,P2_2poignet_D_nsteps,P2_2poignet_G_MQI_mm,P2_2poignet_G_nsteps,P2_MQI_WRISTS_mm,P2_2Tempe_D_MQI_mm,P2_2Tempe_D_nsteps,P2_2Temps_G_MQI_mm,P2_2Temps_G_nsteps,P2_MQI_HEAD_mm
0,/Users/matysprecloux/Desktop/Master IEAP/Code ...,SEATEDD01.csv,11257.605164,18011,5086.439843,18011,16344.045006,15748.775325,18011,18437.193954,...,6990.478950,18011,4791.810419,18011,11782.289369,10070.014078,18011,10379.521199,18011,20449.535277
1,/Users/matysprecloux/Desktop/Master IEAP/Code ...,SEATEDD08.csv,12391.595997,17999,7032.863643,17999,19424.459640,8247.951477,17999,7904.140425,...,10612.732324,17999,19406.999643,17999,30019.731967,10175.956481,17999,9455.445588,17999,19631.402069
2,/Users/matysprecloux/Desktop/Master IEAP/Code ...,SEMID01.csv,8668.166077,17999,6035.964257,17999,14704.130334,13942.428120,17999,14935.841734,...,7473.133557,17999,8656.559835,17999,16129.693392,8852.532002,17999,9113.597368,17999,17966.129370
3,/Users/matysprecloux/Desktop/Master IEAP/Code ...,STANDINGD01.csv,21176.211919,17999,18263.692630,17999,39439.904549,15582.855481,17999,17012.960766,...,23389.330425,17999,22248.213123,17999,45637.543548,13910.023848,17999,13998.875022,17999,27908.898870
